# Check multiple unit options in base case

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

# Add path to "classes" directory:
import sys
sys.path.append('../')

In [2]:
dir_output = 'output'
file_geo_outcomes = 'results_geo_outcomes_'  # + scenario.csv

In [5]:
col_admissions = 'Admissions_2025_Est'
scen_geo_name = 'base'

## Load data

In [6]:
# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
cols_options = ['nearest_ivt_unit_multiple_same_time_and_distance_options',
                'nearest_mt_unit_multiple_same_time_and_distance_options',
                'transfer_unit_multiple_same_time_and_distance_options']
converters = dict()
for col in cols_options:
    converters[col] = lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()
        if len(x) > 0 else np.nan
    )

f = f'{file_geo_outcomes}{scen_geo_name}.csv'
p = os.path.join('..', dir_output, f)
results = pd.read_csv(p, index_col=0, converters=converters)

/tmp/ipykernel_17706/3449109073.py:15: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  results = pd.read_csv(p, index_col=0, converters=converters)


In [7]:
results.columns

Index(['nearest_ivt_unit', 'nearest_ivt_adjusted_time',
       'nearest_ivt_unit_multiple_same_time',
       'nearest_ivt_unit_multiple_same_time_and_distance',
       'nearest_ivt_unit_multiple_same_time_and_distance_options',
       'nearest_ivt_time', 'nearest_ivt_unit_no_bias',
       'nearest_ivt_time_no_bias', 'nearest_mt_time', 'nearest_mt_unit',
       'nearest_mt_unit_multiple_same_time',
       'nearest_mt_unit_multiple_same_time_and_distance',
       'nearest_mt_unit_multiple_same_time_and_distance_options',
       'transfer_time', 'transfer_unit', 'transfer_unit_multiple_same_time',
       'transfer_unit_multiple_same_time_and_distance',
       'transfer_unit_multiple_same_time_and_distance_options',
       'transfer_required', 'area_lower', 'Admissions', 'England',
       'Admissions_2025_Est', 'drip_ship_ivt_time', 'drip_ship_mt_time',
       'drip_ship_mt_travel_time', 'drip_ship_mt_travel_time_with_delay',
       'nlvo_no_treatment_mrs_0-2', 'nlvo_no_treatment_utility',

## Check affected LSOA

Pick out the columns that record whether multiple units are available:

In [8]:
cols_multiple_nearest_units = [
    c for c in results.columns if 'multiple_same' in c]
cols_multiple_nearest_units_flags = [c for c in cols_multiple_nearest_units
                                     if 'options' not in c]

cols_multiple_nearest_units

['nearest_ivt_unit_multiple_same_time',
 'nearest_ivt_unit_multiple_same_time_and_distance',
 'nearest_ivt_unit_multiple_same_time_and_distance_options',
 'nearest_mt_unit_multiple_same_time',
 'nearest_mt_unit_multiple_same_time_and_distance',
 'nearest_mt_unit_multiple_same_time_and_distance_options',
 'transfer_unit_multiple_same_time',
 'transfer_unit_multiple_same_time_and_distance',
 'transfer_unit_multiple_same_time_and_distance_options']

Check how many LSOA have multiple options for their nearest unit.

In [9]:
unit_types = ['nearest_ivt_unit', 'nearest_mt_unit', 'transfer_unit']
for unit_type in unit_types:
    cols = [c for c in cols_multiple_nearest_units if c.startswith(unit_type)]
    df_here = results[cols]
    # Convert list columns to string so they work with value_counts():
    cols_list = [c for c in cols if 'options' in c]
    df_here[cols_list] = df_here[cols_list].astype(str)
    # Rename columns for display:
    rename_dict = dict([(c, c.replace(f'{unit_type}_', '')) for c in cols])
    df_here = df_here.rename(columns=rename_dict)
    # Display value counts:
    print(unit_type)
    display(df_here.value_counts(dropna=False))

nearest_ivt_unit


multiple_same_time  multiple_same_time_and_distance  multiple_same_time_and_distance_options
NaN                 NaN                              NaN                                        32253
1.0                 0.0                              []                                           584
                    1.0                              ['SW170QT', 'SW66SX']                          2
                                                     ['E11BB', 'NW12BU']                            1
                                                     ['BL97TD', 'SK27JE']                           1
                                                     ['E11BB', 'SE59RW']                            1
                                                     ['NW12BU', 'SW66SX']                           1
Name: count, dtype: int64

nearest_mt_unit


multiple_same_time  multiple_same_time_and_distance  multiple_same_time_and_distance_options
NaN                 NaN                              NaN                                        31920
1.0                 0.0                              []                                           918
                    1.0                              ['SW170QT', 'SW66SX']                          2
                                                     ['E11BB', 'NW12BU']                            1
                                                     ['E11BB', 'SE59RW']                            1
                                                     ['NW12BU', 'SW66SX']                           1
Name: count, dtype: int64

transfer_unit


multiple_same_time  multiple_same_time_and_distance  multiple_same_time_and_distance_options
NaN                 NaN                              NaN                                        32146
1.0                 0.0                              []                                           697
Name: count, dtype: int64

For all unit types, nearly all LSOA (out of 32,843 total) have only one nearest unit. This includes the 0.1 minute travel time bias for MT units, so there could be more units that have multiple nearest units but only one of which provides MT.

Pick out LSOA with multiple options:

In [10]:
for unit_type in unit_types:
    cols = [c for c in cols_multiple_nearest_units if c.startswith(unit_type)]
    col_list = [c for c in cols if 'options' in c][0]
    # Pick out rows where the list is not empty ([] has string length 2):
    rows_not_nan = results[col_list].dropna()
    index_not_empty = rows_not_nan[rows_not_nan.str.len() > 0].index
    m = results.index.isin(index_not_empty)
    # Only show these columns:
    cols_to_show = [unit_type, 'transfer_required', col_list, col_admissions]
    # Show:
    print(unit_type)
    display(results.loc[m, cols_to_show].rename(columns={col_list: 'options'}))

nearest_ivt_unit


,nearest_ivt_unit,transfer_required,options,Admissions_2025_Est
LSOA,,,,
ISLINGTON012E,E11BB,False,"[E11BB, NW12BU]",3.36165
KINGSTONUPONTHAMES006D,SW170QT,False,"[SW170QT, SW66SX]",2.24110
OLDHAM026F,BL97TD,True,"[BL97TD, SK27JE]",1.49403
SOUTHWARK009A,E11BB,False,"[E11BB, SE59RW]",0.11205
WANDSWORTH020D,SW170QT,False,"[SW170QT, SW66SX]",1.12055
WESTMINSTER023C,NW12BU,False,"[NW12BU, SW66SX]",0.37348


nearest_mt_unit


,nearest_mt_unit,transfer_required,options,Admissions_2025_Est
LSOA,,,,
ISLINGTON012E,E11BB,False,"[E11BB, NW12BU]",3.36165
KINGSTONUPONTHAMES006D,SW170QT,False,"[SW170QT, SW66SX]",2.24110
SOUTHWARK009A,E11BB,False,"[E11BB, SE59RW]",0.11205
WANDSWORTH020D,SW170QT,False,"[SW170QT, SW66SX]",1.12055
WESTMINSTER023C,NW12BU,False,"[NW12BU, SW66SX]",0.37348


transfer_unit


,transfer_unit,transfer_required,options,Admissions_2025_Est
LSOA,,,,


Compare with total number of LSOA and admissions:

In [11]:
print('Total LSOA:', len(results))
print('Total admissions:', results[col_admissions].sum())

Total LSOA: 32843
Total admissions: 90756.01492
